<a href="https://colab.research.google.com/github/BhavyaJain7/Hybrid-Neural-Retrieval/blob/main/Hybrid_Neural_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install datasets sentence-transformers faiss-cpu rank-bm25 scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from tqdm import tqdm

# libraries for retrieval of data
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
dataset = load_dataset("ms_marco", "v1.1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

v1.1/validation-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

v1.1/train-00000-of-00001.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

v1.1/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

In [5]:
dataset

DatasetDict({
    validation: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 10047
    })
    train: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 82326
    })
    test: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 9650
    })
})

MS Macro dataset is split into fixed benchmark splits used for the research purposes as -:
Train : ~808,731 queries
Validation : ~101,093 queries
Test : ~100,000 queries
Ratio : 80-10-10

In [6]:
queries = []
corpus = []
qrels = []

for item in dataset["validation"]:

    query = item["query"]
    passages = item["passages"]["passage_text"]
    labels = item["passages"]["is_selected"]

    queries.append(query)

    for p, l in zip(passages, labels):
        corpus.append(p)
        qrels.append(l)

In [7]:
queries_df = pd.DataFrame({
    "query_id": range(len(queries)),
    "query": queries
})

corpus_df = pd.DataFrame({
    "doc_id": range(len(corpus)),
    "text": corpus
})

labels_df = pd.DataFrame({
    "doc_id": range(len(qrels)),
    "relevant": qrels
})

In [8]:
print("Number of queries:", len(queries_df))
print("Number of documents:", len(corpus_df))
print("Relevant passages:", labels_df['relevant'].sum())

Number of queries: 10047
Number of documents: 82360
Relevant passages: 10783


In [9]:
tokenized_corpus = [doc.split(" ") for doc in corpus_df["text"]]

tokenized courpus is created to be utilized for BM25 retrieval

In [10]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english", max_features=50000)

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_df["text"])

TF-IDF corpus

In [11]:
queries_df.to_csv("queries.csv", index=False)
corpus_df.to_csv("corpus.csv", index=False)
labels_df.to_csv("labels.csv", index=False)

In [12]:
#converting the dataset into standard IR format
corpus = {}
queries = {}
qrels = {}


doc_id = 0

for q_id, item in enumerate(dataset["validation"]):

    queries[q_id] = item["query"]

    passages = item["passages"]["passage_text"]
    labels = item["passages"]["is_selected"]

    for p, l in zip(passages, labels):

        corpus[doc_id] = p

        if l == 1:
            if q_id not in qrels:
                qrels[q_id] = []
            qrels[q_id].append(doc_id)

        doc_id += 1

In [13]:
print("Total queries:", len(queries))
print("Total documents:", len(corpus))
print("Queries with relevant docs:", len(qrels))

Total queries: 10047
Total documents: 82360
Queries with relevant docs: 9706


In [14]:
# TF-IDF Retrieval

import time
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Preparing corpus and queries...")

doc_ids = list(corpus.keys())
documents = list(corpus.values())

query_ids = list(queries.keys())
query_texts = list(queries.values())

print("Total Documents:", len(documents))
print("Total Queries:", len(query_texts))


# Build TF-IDF index
print("\nBuilding TF-IDF index...")

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000
)

doc_vectors = tfidf_vectorizer.fit_transform(documents)

print("TF-IDF Matrix Shape:", doc_vectors.shape)


# Retrieval function
def retrieve_tfidf(query, top_k=10):

    query_vec = tfidf_vectorizer.transform([query])

    similarities = cosine_similarity(query_vec, doc_vectors).flatten()

    top_indices = similarities.argsort()[-top_k:][::-1]

    return [doc_ids[i] for i in top_indices]


# Run retrieval for all queries
print("\nRunning retrieval...")

results = {}
top_k = 10

start_time = time.time()

for q_id, query in tqdm(queries.items()):

    retrieved_docs = retrieve_tfidf(query, top_k)

    results[q_id] = retrieved_docs

end_time = time.time()

latency = (end_time - start_time) / len(queries)


# Evaluation metrics

def recall_at_k(results, qrels, k=10):

    recalls = []

    for q_id, retrieved in results.items():

        if q_id not in qrels:
            continue

        relevant_docs = set(qrels[q_id])
        retrieved_k = set(retrieved[:k])

        recall = len(relevant_docs & retrieved_k) / len(relevant_docs)

        recalls.append(recall)

    return np.mean(recalls)


def mrr_at_k(results, qrels, k=10):

    mrr_scores = []

    for q_id, retrieved in results.items():

        if q_id not in qrels:
            continue

        relevant_docs = set(qrels[q_id])

        score = 0

        for rank, doc_id in enumerate(retrieved[:k], start=1):

            if doc_id in relevant_docs:
                score = 1 / rank
                break

        mrr_scores.append(score)

    return np.mean(mrr_scores)


print("\nEvaluating results...")

recall = recall_at_k(results, qrels, k=10)
mrr = mrr_at_k(results, qrels, k=10)


print("\n===== TF-IDF RESULTS =====")

print("Recall@10:", recall)
print("MRR@10:", mrr)
print("Average Query Latency:", latency, "seconds")

Preparing corpus and queries...
Total Documents: 82360
Total Queries: 10047

Building TF-IDF index...
TF-IDF Matrix Shape: (82360, 50000)

Running retrieval...


100%|██████████| 10047/10047 [09:19<00:00, 17.94it/s]



Evaluating results...

===== TF-IDF RESULTS =====
Recall@10: 0.6493543512603888
MRR@10: 0.3043432879024266
Average Query Latency: 0.05573095138172397 seconds


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##TF-IDF Workflow
1. Conversion of document corpus (dataset) into numerical vectors using TF-IDF vectorization.
2. Similarly, user Query will be converted into TF-IDF vector
3. Similarities are computed using cosine similarity to identify most relevant document.
4. System retrieves top 10 most relevant documents for each query based on these scores.
5. Performance is evaluated using metrics like Recall@10, Mean Reciprocal Rank @10, Average Latency.



In [15]:
# BM25 Retrieval

import time
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi


print("Preparing corpus...")

doc_ids = list(corpus.keys())
documents = list(corpus.values())

query_ids = list(queries.keys())
query_texts = list(queries.values())

print("Total Documents:", len(documents))
print("Total Queries:", len(query_texts))


# Tokenize the corpus
tokenized_corpus = [doc.lower().split() for doc in documents]

print("\nBuilding BM25 index...")

bm25 = BM25Okapi(tokenized_corpus)


# Retrieval function
def retrieve_bm25(query, top_k=10):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[-top_k:][::-1]

    return [doc_ids[i] for i in top_indices]


# Run retrieval
print("\nRunning BM25 retrieval...")

results = {}
top_k = 10

start_time = time.time()

for q_id, query in tqdm(queries.items()):

    retrieved_docs = retrieve_bm25(query, top_k)

    results[q_id] = retrieved_docs

end_time = time.time()

latency = (end_time - start_time) / len(queries)


# Evaluation metrics

def recall_at_k(results, qrels, k=10):

    recalls = []

    for q_id, retrieved in results.items():

        if q_id not in qrels:
            continue

        relevant_docs = set(qrels[q_id])
        retrieved_k = set(retrieved[:k])

        recall = len(relevant_docs & retrieved_k) / len(relevant_docs)

        recalls.append(recall)

    return np.mean(recalls)


def mrr_at_k(results, qrels, k=10):

    mrr_scores = []

    for q_id, retrieved in results.items():

        if q_id not in qrels:
            continue

        relevant_docs = set(qrels[q_id])

        score = 0

        for rank, doc_id in enumerate(retrieved[:k], start=1):

            if doc_id in relevant_docs:
                score = 1 / rank
                break

        mrr_scores.append(score)

    return np.mean(mrr_scores)


print("\nEvaluating BM25 results...")

recall = recall_at_k(results, qrels, k=10)
mrr = mrr_at_k(results, qrels, k=10)


print("\n===== BM25 RESULTS =====")

print("Recall@10:", recall)
print("MRR@10:", mrr)
print("Average Query Latency:", latency, "seconds")

Preparing corpus...
Total Documents: 82360
Total Queries: 10047

Building BM25 index...

Running BM25 retrieval...


100%|██████████| 10047/10047 [34:45<00:00,  4.82it/s]


Evaluating BM25 results...

===== BM25 RESULTS =====
Recall@10: 0.6545744900061818
MRR@10: 0.3328086619633086
Average Query Latency: 0.2075325027204597 seconds


#BM25 Retrieval
1. Dataset/document corpus is prepared and tokenized.
2. BM25 index is built using this corpus to calculate the term based relevance scores.
3. Similarly, the query is also tokenized and compared against indexed documents using the same BM25 index function
4. System then ranks the top 10 most relevant documnets using relevance scores.
5. Metrics used - Recall@10, MRR@10 and Average Latency

In [17]:
# Dense Retrieval

import time
import numpy as np
from tqdm import tqdm
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Checking GPU availability...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device being used:", device)
print("\nLoading embedding model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

doc_ids = list(corpus.keys())
documents = list(corpus.values())

query_items = list(queries.items())[:1000]

print("\nDocuments:", len(documents))
print("Queries used for evaluation:", len(query_items))

print("\nEncoding documents (GPU)...")

doc_embeddings = model.encode(
    documents,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Document embedding shape:", doc_embeddings.shape)

# Retrieval function
def retrieve_dense(query, top_k=10):
    query_embedding = model.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, doc_embeddings).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]
    return [doc_ids[i] for i in top_indices]

# Run retrieval
print("\nRunning dense retrieval...")

results = {}
top_k = 10

start_time = time.time()

for q_id, query in tqdm(query_items):
    retrieved_docs = retrieve_dense(query, top_k)
    results[q_id] = retrieved_docs

end_time = time.time()
latency = (end_time - start_time) / len(query_items)

# Evaluation metrics
def recall_at_k(results, qrels, k=10):
    recalls = []
    for q_id, retrieved in results.items():
        if q_id not in qrels:
            continue
        relevant_docs = set(qrels[q_id])
        retrieved_k = set(retrieved[:k])
        recall = len(relevant_docs & retrieved_k) / len(relevant_docs)
        recalls.append(recall)
    return np.mean(recalls)

def mrr_at_k(results, qrels, k=10):
    mrr_scores = []
    for q_id, retrieved in results.items():
        if q_id not in qrels:
            continue
        relevant_docs = set(qrels[q_id])
        score = 0
        for rank, doc_id in enumerate(retrieved[:k], start=1):
            if doc_id in relevant_docs:
                score = 1 / rank
                break
        mrr_scores.append(score)
    return np.mean(mrr_scores)

print("\nEvaluating dense retrieval...")
recall = recall_at_k(results, qrels, k=10)
mrr = mrr_at_k(results, qrels, k=10)
print("\n===== DENSE RETRIEVAL RESULTS (1000 QUERIES) =====")
print("Recall@10:", recall)
print("MRR@10:", mrr)
print("Average Query Latency:", latency, "seconds")

Checking GPU availability...
Device being used: cuda

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Documents: 82360
Queries used for evaluation: 1000

Encoding documents (GPU)...


Batches:   0%|          | 0/644 [00:00<?, ?it/s]

Document embedding shape: (82360, 384)

Running dense retrieval...


100%|██████████| 1000/1000 [02:44<00:00,  6.08it/s]


Evaluating dense retrieval...

===== DENSE RETRIEVAL RESULTS (1000 QUERIES) =====
Recall@10: 0.8933126508100654
MRR@10: 0.5027892680684821
Average Query Latency: 0.16440525960922242 seconds
